In [1]:
recording_duration = 3600;  #chb01_03 recording duration in seconds
window_size = 5;  #window size in seconds

num_windows = recording_duration / window_size
print("Number of windows:", num_windows)

Number of windows: 720.0


In [2]:
for i in range(int(num_windows)):
    start_time  = i * window_size
    end_time = start_time + window_size 
    print("Window", i, ":", start_time, " to ", end_time)

Window 0 : 0  to  5
Window 1 : 5  to  10
Window 2 : 10  to  15
Window 3 : 15  to  20
Window 4 : 20  to  25
Window 5 : 25  to  30
Window 6 : 30  to  35
Window 7 : 35  to  40
Window 8 : 40  to  45
Window 9 : 45  to  50
Window 10 : 50  to  55
Window 11 : 55  to  60
Window 12 : 60  to  65
Window 13 : 65  to  70
Window 14 : 70  to  75
Window 15 : 75  to  80
Window 16 : 80  to  85
Window 17 : 85  to  90
Window 18 : 90  to  95
Window 19 : 95  to  100
Window 20 : 100  to  105
Window 21 : 105  to  110
Window 22 : 110  to  115
Window 23 : 115  to  120
Window 24 : 120  to  125
Window 25 : 125  to  130
Window 26 : 130  to  135
Window 27 : 135  to  140
Window 28 : 140  to  145
Window 29 : 145  to  150
Window 30 : 150  to  155
Window 31 : 155  to  160
Window 32 : 160  to  165
Window 33 : 165  to  170
Window 34 : 170  to  175
Window 35 : 175  to  180
Window 36 : 180  to  185
Window 37 : 185  to  190
Window 38 : 190  to  195
Window 39 : 195  to  200
Window 40 : 200  to  205
Window 41 : 205  to  210
Wi

In [3]:
sfreq = 256 
for i in range(5):
    start_time = i * window_size
    end_time = start_time + window_size

    start_sample = int(start_time * sfreq)
    end_sample = int(end_time * sfreq)
    print("Window", i, ":", start_time, " to ", end_time, "| ", end="")
    print("Samples:", start_sample, " to ", end_sample)

Window 0 : 0  to  5 | Samples: 0  to  1280
Window 1 : 5  to  10 | Samples: 1280  to  2560
Window 2 : 10  to  15 | Samples: 2560  to  3840
Window 3 : 15  to  20 | Samples: 3840  to  5120
Window 4 : 20  to  25 | Samples: 5120  to  6400


In [12]:
windows = []

seizure_start = 2996
seizure_end = 3036

for i in range(int(num_windows)):
    start_time = i * window_size
    end_time = start_time + window_size

    if start_time >= seizure_start and end_time <= seizure_end:
        label = "Seizure"

    elif end_time <= seizure_start or start_time >= seizure_end:
        label = "Non-Seizure"

    else:
        label = "Partial Seizure"

    if label != "Partial Seizure":
        windows.append({
            "window_index": i,
            "start_time": start_time,
            "end_time": end_time,
            "label": label
        })

print(len(windows))


718


In [16]:
import pandas as pd
windows_df = pd.DataFrame(windows)
windows_df.head()
windows_df["label"].value_counts()

label
Non-Seizure    711
Seizure          7
Name: count, dtype: int64

In [17]:
seizure_windows = windows_df[windows_df["label"] == "Seizure"]
print("Seizure windows:", len(seizure_windows))


Seizure windows: 7


In [20]:
#Now I know where the metadata locations are for the seizure windows 
#Next steps are to extract the actual EEG values and calculate features 
from pathlib import Path
import mne

project_root = Path.cwd().parent

eeg_file = project_root / "data" / "raw" / "chb01" / "chb01_03.edf"

print(eeg_file)
print("Exists:", eeg_file.exists()) #checks if chb01_03 can be read

raw = mne.io.read_raw_edf(
    eeg_file,
    preload=False,
    verbose=False
)


sfreq = raw.info["sfreq"]

print(raw)
print("Sampling frequency:", sfreq)

c:\seizure-detection-ML-project\data\raw\chb01\chb01_03.edf
Exists: True
<RawEDF | chb01_03.edf, 23 x 921600 (3600.0 s), ~22 KiB, data not loaded>
Sampling frequency: 256.0


C:\Users\rahma\AppData\Local\Temp\ipykernel_22320\3555653787.py:13: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(


In [25]:
first_s_window = seizure_windows.iloc[0]
print(first_s_window)
start_sample = int(first_s_window["start_time"] * sfreq)
end_sample = int(first_s_window["end_time"] * sfreq)
print("Start Sample:", start_sample)
print("End Sample:", end_sample)

window_index        600
start_time         3000
end_time           3005
label           Seizure
Name: 599, dtype: object
Start Sample: 768000
End Sample: 769280


In [29]:

data, times = raw.get_data(picks="FP1-F7", start=start_sample, stop=end_sample, return_times=True)
print(data.shape, times.shape)

(1, 1280) (1280,)


In [30]:
data_uv = data[0] * 1e6
print(data_uv.shape)
print(data_uv[:5])

(1280,)
[-45.51892552 -52.55189255 -56.45909646 -51.77045177 -67.78998779]


In [33]:
for index, row in windows_df.head(3).iterrows():
    print("DataFrame index:", index)
    print("EEG window index:", row["window_index"])
    print("Start time:", row["start_time"])
    print("Label:", row["label"])
    print()

DataFrame index: 0
EEG window index: 0
Start time: 0
Label: Non-Seizure

DataFrame index: 1
EEG window index: 1
Start time: 5
Label: Non-Seizure

DataFrame index: 2
EEG window index: 2
Start time: 10
Label: Non-Seizure



In [ ]:
for index, row in seizure_windows.head(3).iterrows():
    start_sample = int(row["start_time"] * sfreq)
    end_sample = int(row["end_time"] * sfreq)
    data, times = raw.get_data(picks="FP1-F7", start=start_sample, stop=end_sample, return_times=True)
    data_uv = data[0] * 1e6
    print("Window: ", row["window_index"])
    print("Label: ")